## 데이터 구축 방안

1. 시군구 코드, 용도 코드 변환 규칙 작성
    1. 시군구(주민등록인구와 일치) 단위 집계
        1. 변환 규칙 1: 오류만 정정, 변환 안함
        2. 변환 규칙 2: 자치구 단위 집계, 자치구 단위 통계 비교 목적으로 사용
    2. 용도 통계분류(5개) 단위 집계를 위한 용도 변환 규칙 작성
2. 층별개요 데이터 정제
    1. 시군구*코드, 층*구분*코드, 주*용도\_코드, 면적(㎡) 컬럼의 값이 유효한 경우만 추출
3. 층별개요와 표제부 데이터 연계 및 정제
    1. 표제부 데이터 정제
        1. 표제부에서 주건축물만 추출(부속건축물, 미기재 제거)
    2. 표제부 PK (관리\_건축물대장\_PK) 사용하여 층별개요에 표제부 데이터 연계
        1. inner join 층별개요에 유효한 주건축물 표제부 PK가 있으면 n:1 연계
    3. 표제부 연면적이 0보다 크면서, 층 면적이 연면적보다 큰 경우 제거
4. 시군구(주민등록인구와 일치), 용도(통계분류, 5개) 단위 집계


In [5]:
import polars as pl
from pathlib import Path
from pprint import pprint

df_master = pl.scan_parquet("data/건축물대장_기본개요_2022년_12월/*")
df_dong = pl.scan_parquet("data/건축물대장_표제부_2022년_12월/*")
df_floor = pl.scan_parquet("data/건축물대장_층별개요_2022년_12월/*")

df_sgg = pl.scan_csv("data/code_sgg.csv", dtypes={"시군구코드": str})

## 시군구, 용도 변환 규칙


생산량(허가, 착공, 준공) 집계 기준

-   가설건축물, 기타 용도 제외

3. 연면적 : 층 용도별 면적의 합
4. 용도 : 층별 용도

-   층 용도별 면적의 합
-   건축인허가
    -   면적은 층별 층면적 사용
    -   용도는 층별 주용도 사용
    -   시군구는 층별 사용(어차피 기본에서 온 것)
-   주택인허가
    -   면적은 층별 층면적 사용
    -   용도는 층별 용도 사용
    -   시군구는 층별 사용(어차피 기본에서 온 것)

---

건축물 : 토지에 정착하는 공작물중 지붕과 기둥 또는 벽이 있는 것과 이에 부수되는 시설물, 지하 또는 고가의 공작물에 설치하는 사무소, 공연장, 점포, 차고, 창고 기타 대통령령이 정하는 것

건축물의 용도 : 건축물의 종류(단독주택, 공동주택 등)를 유사한 구조, 이용목적 및 형태별로 묶어 분류한 것을 말함

주거용 : 공동주택, 단독주택

상업용 : 판매및영업시설,제1종근린생활시설,제2종근린생활시설,판매시설,운수시설,업무시설,숙박시설,위락시설,위험물저장및처리시설,자동차관련시설,야영장시설

농수산용 : 동,식물관련시설

공업용 : 공장

공공용 : 공공용시설,교정및군사시설,방송통신시설,발전시설

교육및사회용 : 교육연구및복지시설,문화및집회시설,종교시설,의료시설,교육연구시설,노유자시설,수련시설,운동시설,묘지관련시설,관광휴게시설,장례시설

기타 : 기타,창고시설,분뇨.쓰레기처리시설,가설건축물시설,자원순환관련시설

https://www.index.go.kr/unity/potal/main/EachDtlPageDetail.do?idx_cd=1226

---

※ 위 자료에서 분류하는 통계용 건축물의 용도는 아래와 같습니다.

[용도별 건축물 분류(통계용)]

-   주거용 : 단독, 다가구, 아파트, 연립, 다세대, 기타(다중주택, 공관, 기숙사 등)

-   상업용 : 근린생활, 판매, 업무, 숙박, 위락, 운수, 자동차관련시설 등

-   공업용 : 공장

-   교육및사회용 : 문화집회시설(극장 등), 종교시설, 의료시설, 교육연구시설(학교 등), 노유자시설, 수련시설, 운동시설, 관광휴게시설, 묘지관련시설 및 장례시설 등

-   기타 : 농수산용(축사, 온실), 공공용(공공청사, 방송국), 창고 등

[상업용 건축물 분류(통계용)]

-   제1종근린생활시설 : 소매점, 휴게음식점, 이용원, 의원 등

-   제2종근린생활시설 : 공연장, 금융업소, 제조업소, 고시원 등

-   판매시설 : 도매시장, 소매시장, 상점 등

-   업무시설 : 공공업무시설, 일반업무시설(사무소, 오피스텔 등)

-   기타 : 위락시설, 숙박시설, 운수시설, 자동차관련시설 등

[출처] 전국 건축물 총 7,354,340동… 연면적 41억 3천만㎡|작성자 국토교통부
https://blog.naver.com/mltmkr/223032348016


In [6]:
use_agg = {
    "01000": "주거용",
    "01001": "주거용",
    "01002": "주거용",
    "01003": "주거용",
    "01004": "주거용",
    "02000": "주거용",
    "02001": "주거용",
    "02002": "주거용",
    "02003": "주거용",
    "02004": "상업용",
    "02005": "기타",
    "02006": "기타",
    "02007": "주거용",
    "02100": "주거용",
    "02101": "주거용",
    "02102": "주거용",
    "03000": "상업용",
    "03001": "상업용",
    "03002": "상업용",
    "03003": "상업용",
    "03004": "상업용",
    "03005": "상업용",
    "03006": "상업용",
    "03007": "상업용",
    "03008": "상업용",
    "03009": "상업용",
    "03010": "상업용",
    "03011": "상업용",
    "03012": "상업용",
    "03013": "상업용",
    "03014": "상업용",
    "03015": "상업용",
    "03016": "상업용",
    "03017": "상업용",
    "03018": "상업용",
    "03019": "상업용",
    "03020": "상업용",
    "03021": "상업용",
    "03022": "상업용",
    "03023": "상업용",
    "03024": "상업용",
    "03025": "상업용",
    "03026": "상업용",
    "03027": "상업용",
    "03028": "상업용",
    "03029": "상업용",
    "03030": "상업용",
    "03031": "상업용",
    "03032": "상업용",
    "03033": "상업용",
    "03034": "상업용",
    "03035": "상업용",
    "03036": "상업용",
    "03037": "상업용",
    "03038": "상업용",
    "03100": "상업용",
    "03101": "상업용",
    "03102": "상업용",
    "03103": "상업용",
    "03104": "상업용",
    "03105": "상업용",
    "03106": "기타",
    "03107": "상업용",
    "03108": "상업용",
    "03109": "상업용",
    "03110": "상업용",
    "03111": "상업용",
    "03112": "상업용",
    "03113": "상업용",
    "03114": "상업용",
    "03115": "상업용",
    "03199": "상업용",
    "03200": "상업용",
    "03201": "상업용",
    "03202": "상업용",
    "03203": "상업용",
    "03204": "상업용",
    "03205": "상업용",
    "03299": "상업용",
    "03300": "상업용",
    "03999": "상업용",
    "04000": "상업용",
    "04001": "상업용",
    "04002": "상업용",
    "04003": "상업용",
    "04004": "상업용",
    "04005": "상업용",
    "04006": "상업용",
    "04007": "상업용",
    "04008": "상업용",
    "04009": "상업용",
    "04010": "상업용",
    "04011": "상업용",
    "04012": "상업용",
    "04013": "교육및사회용",
    "04014": "상업용",
    "04015": "상업용",
    "04016": "상업용",
    "04017": "상업용",
    "04018": "상업용",
    "04019": "상업용",
    "04020": "상업용",
    "04021": "상업용",
    "04022": "상업용",
    "04023": "상업용",
    "04024": "상업용",
    "04025": "상업용",
    "04026": "상업용",
    "04027": "상업용",
    "04028": "상업용",
    "04029": "상업용",
    "04030": "상업용",
    "04031": "상업용",
    "04032": "상업용",
    "04033": "상업용",
    "04034": "상업용",
    "04035": "상업용",
    "04036": "상업용",
    "04037": "상업용",
    "04038": "상업용",
    "04039": "상업용",
    "04040": "상업용",
    "04041": "상업용",
    "04042": "상업용",
    "04043": "상업용",
    "04044": "상업용",
    "04045": "상업용",
    "04046": "상업용",
    "04047": "상업용",
    "04048": "상업용",
    "04049": "상업용",
    "04050": "상업용",
    "04100": "상업용",
    "04101": "상업용",
    "04102": "상업용",
    "04103": "상업용",
    "04104": "상업용",
    "04105": "상업용",
    "04106": "상업용",
    "04107": "상업용",
    "04108": "상업용",
    "04109": "상업용",
    "04199": "상업용",
    "04200": "상업용",
    "04201": "상업용",
    "04202": "상업용",
    "04203": "상업용",
    "04204": "상업용",
    "04205": "상업용",
    "04206": "상업용",
    "04207": "상업용",
    "04208": "상업용",
    "04299": "상업용",
    "04300": "상업용",
    "04301": "상업용",
    "04302": "상업용",
    "04303": "상업용",
    "04304": "상업용",
    "04305": "상업용",
    "04306": "상업용",
    "04307": "상업용",
    "04308": "상업용",
    "04399": "상업용",
    "04400": "상업용",
    "04401": "상업용",
    "04402": "상업용",
    "04403": "상업용",
    "04404": "상업용",
    "04405": "상업용",
    "04406": "상업용",
    "04499": "상업용",
    "04999": "상업용",
    "05000": "교육및사회용",
    "05100": "교육및사회용",
    "05101": "교육및사회용",
    "05102": "교육및사회용",
    "05103": "교육및사회용",
    "05104": "교육및사회용",
    "05105": "교육및사회용",
    "05106": "교육및사회용",
    "05107": "교육및사회용",
    "05108": "교육및사회용",
    "05199": "교육및사회용",
    "05200": "교육및사회용",
    "05201": "교육및사회용",
    "05202": "교육및사회용",
    "05203": "교육및사회용",
    "05204": "교육및사회용",
    "05205": "교육및사회용",
    "05299": "교육및사회용",
    "05300": "교육및사회용",
    "05301": "교육및사회용",
    "05302": "교육및사회용",
    "05303": "교육및사회용",
    "05304": "교육및사회용",
    "05305": "교육및사회용",
    "05306": "교육및사회용",
    "05399": "교육및사회용",
    "05400": "교육및사회용",
    "05401": "교육및사회용",
    "05402": "교육및사회용",
    "05403": "교육및사회용",
    "05404": "교육및사회용",
    "05405": "교육및사회용",
    "05406": "교육및사회용",
    "05407": "교육및사회용",
    "05408": "교육및사회용",
    "05499": "교육및사회용",
    "05500": "교육및사회용",
    "05501": "교육및사회용",
    "05502": "교육및사회용",
    "05503": "교육및사회용",
    "05599": "교육및사회용",
    "05999": "교육및사회용",
    "06000": "교육및사회용",
    "06100": "교육및사회용",
    "06101": "교육및사회용",
    "06102": "교육및사회용",
    "06103": "교육및사회용",
    "06104": "교육및사회용",
    "06105": "교육및사회용",
    "06106": "교육및사회용",
    "06107": "교육및사회용",
    "06108": "교육및사회용",
    "06109": "교육및사회용",
    "06110": "교육및사회용",
    "06199": "교육및사회용",
    "06999": "교육및사회용",
    "07000": "상업용",
    "07001": "상업용",
    "07100": "상업용",
    "07101": "상업용",
    "07102": "상업용",
    "07103": "상업용",
    "07104": "상업용",
    "07105": "상업용",
    "07199": "상업용",
    "07200": "상업용",
    "07201": "상업용",
    "07202": "상업용",
    "07203": "상업용",
    "07204": "상업용",
    "07205": "상업용",
    "07206": "상업용",
    "07207": "상업용",
    "07208": "상업용",
    "07209": "상업용",
    "07210": "상업용",
    "07211": "상업용",
    "07212": "상업용",
    "07300": "상업용",
    "07301": "상업용",
    "07302": "상업용",
    "07399": "상업용",
    "07999": "상업용",
    "08000": "상업용",
    "08001": "상업용",
    "08002": "상업용",
    "08003": "상업용",
    "08004": "상업용",
    "08005": "상업용",
    "08006": "상업용",
    "08007": "상업용",
    "08008": "상업용",
    "08999": "상업용",
    "09000": "교육및사회용",
    "09100": "교육및사회용",
    "09101": "교육및사회용",
    "09102": "교육및사회용",
    "09103": "교육및사회용",
    "09104": "교육및사회용",
    "09105": "교육및사회용",
    "09106": "교육및사회용",
    "09107": "교육및사회용",
    "09108": "교육및사회용",
    "09109": "교육및사회용",
    "09199": "교육및사회용",
    "09200": "교육및사회용",
    "09201": "교육및사회용",
    "09202": "교육및사회용",
    "09299": "교육및사회용",
    "09301": "교육및사회용",
    "09999": "교육및사회용",
    "10000": "교육및사회용",
    "10001": "교육및사회용",
    "10002": "교육및사회용",
    "10003": "교육및사회용",
    "10004": "교육및사회용",
    "10005": "교육및사회용",
    "10100": "교육및사회용",
    "10101": "교육및사회용",
    "10102": "교육및사회용",
    "10103": "교육및사회용",
    "10104": "교육및사회용",
    "10105": "교육및사회용",
    "10106": "교육및사회용",
    "10107": "교육및사회용",
    "10199": "교육및사회용",
    "10200": "교육및사회용",
    "10201": "교육및사회용",
    "10202": "교육및사회용",
    "10299": "교육및사회용",
    "10300": "교육및사회용",
    "10301": "교육및사회용",
    "10302": "교육및사회용",
    "10303": "교육및사회용",
    "10399": "교육및사회용",
    "10400": "교육및사회용",
    "10401": "교육및사회용",
    "10402": "교육및사회용",
    "10999": "교육및사회용",
    "11000": "교육및사회용",
    "11100": "교육및사회용",
    "11101": "교육및사회용",
    "11102": "교육및사회용",
    "11103": "교육및사회용",
    "11104": "교육및사회용",
    "11199": "교육및사회용",
    "11201": "교육및사회용",
    "11202": "교육및사회용",
    "11203": "교육및사회용",
    "11999": "교육및사회용",
    "12000": "교육및사회용",
    "12001": "교육및사회용",
    "12100": "교육및사회용",
    "12101": "교육및사회용",
    "12102": "교육및사회용",
    "12103": "교육및사회용",
    "12104": "교육및사회용",
    "12105": "교육및사회용",
    "12199": "교육및사회용",
    "12200": "교육및사회용",
    "12201": "교육및사회용",
    "12202": "교육및사회용",
    "12203": "교육및사회용",
    "12299": "교육및사회용",
    "12300": "교육및사회용",
    "12301": "교육및사회용",
    "12302": "교육및사회용",
    "12303": "교육및사회용",
    "12304": "교육및사회용",
    "12305": "교육및사회용",
    "12399": "교육및사회용",
    "12999": "교육및사회용",
    "13000": "교육및사회용",
    "13001": "교육및사회용",
    "13002": "교육및사회용",
    "13003": "교육및사회용",
    "13004": "교육및사회용",
    "13005": "교육및사회용",
    "13006": "교육및사회용",
    "13007": "교육및사회용",
    "13008": "교육및사회용",
    "13009": "교육및사회용",
    "13010": "교육및사회용",
    "13011": "교육및사회용",
    "13012": "교육및사회용",
    "13014": "교육및사회용",
    "13100": "교육및사회용",
    "13101": "교육및사회용",
    "13102": "교육및사회용",
    "13103": "교육및사회용",
    "13104": "교육및사회용",
    "13105": "교육및사회용",
    "13106": "교육및사회용",
    "13107": "교육및사회용",
    "13108": "교육및사회용",
    "13109": "교육및사회용",
    "13110": "교육및사회용",
    "13999": "교육및사회용",
    "14000": "상업용",
    "14100": "기타",
    "14101": "기타",
    "14102": "기타",
    "14103": "기타",
    "14199": "기타",
    "14200": "상업용",
    "14201": "상업용",
    "14202": "상업용",
    "14203": "상업용",
    "14204": "상업용",
    "14205": "상업용",
    "14206": "상업용",
    "14299": "상업용",
    "15000": "상업용",
    "15001": "상업용",
    "15002": "상업용",
    "15003": "상업용",
    "15100": "상업용",
    "15101": "상업용",
    "15102": "상업용",
    "15103": "상업용",
    "15104": "상업용",
    "15199": "상업용",
    "15200": "상업용",
    "15201": "상업용",
    "15202": "상업용",
    "15203": "상업용",
    "15204": "상업용",
    "15205": "상업용",
    "15206": "상업용",
    "15207": "상업용",
    "15208": "상업용",
    "15299": "상업용",
    "15300": "상업용",
    "15999": "상업용",
    "16000": "상업용",
    "16001": "상업용",
    "16002": "상업용",
    "16003": "상업용",
    "16004": "상업용",
    "16005": "상업용",
    "16006": "상업용",
    "16007": "상업용",
    "16008": "상업용",
    "16009": "상업용",
    "16010": "상업용",
    "16011": "상업용",
    "16012": "상업용",
    "16013": "상업용",
    "16999": "상업용",
    "17000": "공업용",
    "17100": "공업용",
    "17200": "공업용",
    "17300": "공업용",
    "17301": "공업용",
    "17302": "공업용",
    "17303": "공업용",
    "17304": "공업용",
    "17305": "공업용",
    "17306": "공업용",
    "17307": "공업용",
    "17308": "공업용",
    "17309": "공업용",
    "17999": "공업용",
    "18000": "기타",
    "18001": "기타",
    "18002": "기타",
    "18003": "기타",
    "18004": "기타",
    "18100": "기타",
    "18101": "기타",
    "18102": "기타",
    "18103": "기타",
    "18999": "기타",
    "19000": "상업용",
    "19001": "상업용",
    "19002": "상업용",
    "19003": "공업용",
    "19004": "상업용",
    "19005": "상업용",
    "19006": "상업용",
    "19007": "상업용",
    "19008": "상업용",
    "19009": "상업용",
    "19010": "상업용",
    "19011": "상업용",
    "19012": "상업용",
    "19013": "상업용",
    "19014": "상업용",
    "19015": "상업용",
    "19016": "상업용",
    "19017": "상업용",
    "19018": "상업용",
    "19019": "상업용",
    "19020": "상업용",
    "19021": "상업용",
    "19022": "상업용",
    "19999": "상업용",
    "20000": "상업용",
    "20001": "상업용",
    "20002": "상업용",
    "20003": "상업용",
    "20004": "상업용",
    "20005": "상업용",
    "20006": "상업용",
    "20007": "상업용",
    "20008": "상업용",
    "20009": "상업용",
    "20010": "상업용",
    "20011": "상업용",
    "20999": "상업용",
    "21000": "기타",
    "21001": "기타",
    "21002": "기타",
    "21003": "기타",
    "21004": "기타",
    "21005": "기타",
    "21006": "기타",
    "21100": "기타",
    "21101": "기타",
    "21102": "기타",
    "21103": "기타",
    "21104": "기타",
    "21105": "기타",
    "21106": "기타",
    "21107": "기타",
    "21108": "기타",
    "21200": "기타",
    "21201": "기타",
    "21202": "기타",
    "21203": "기타",
    "21204": "기타",
    "21205": "기타",
    "21206": "기타",
    "21207": "기타",
    "21299": "기타",
    "21999": "기타",
    "22000": "기타",
    "22001": "기타",
    "22002": "기타",
    "22003": "기타",
    "22004": "기타",
    "22005": "기타",
    "22999": "기타",
    "23000": "기타",
    "23001": "기타",
    "23002": "기타",
    "23003": "기타",
    "23004": "기타",
    "23005": "기타",
    "23006": "기타",
    "23007": "기타",
    "23100": "기타",
    "23101": "기타",
    "23102": "기타",
    "23103": "기타",
    "23200": "기타",
    "23201": "기타",
    "23202": "기타",
    "23203": "기타",
    "23999": "기타",
    "24000": "기타",
    "24001": "기타",
    "24002": "기타",
    "24003": "기타",
    "24004": "기타",
    "24005": "기타",
    "24100": "기타",
    "24101": "기타",
    "24102": "기타",
    "24103": "기타",
    "24104": "기타",
    "24105": "기타",
    "24999": "기타",
    "25000": "기타",
    "25001": "기타",
    "25999": "기타",
    "26000": "교육및사회용",
    "26001": "교육및사회용",
    "26002": "교육및사회용",
    "26003": "교육및사회용",
    "26004": "교육및사회용",
    "26005": "교육및사회용",
    "26006": "교육및사회용",
    "26100": "교육및사회용",
    "26101": "교육및사회용",
    "26102": "교육및사회용",
    "26103": "교육및사회용",
    "26999": "교육및사회용",
    "27000": "교육및사회용",
    "27001": "교육및사회용",
    "27002": "교육및사회용",
    "27003": "교육및사회용",
    "27004": "교육및사회용",
    "27005": "교육및사회용",
    "27006": "교육및사회용",
    "27007": "교육및사회용",
    "27008": "교육및사회용",
    "27009": "교육및사회용",
    "27999": "교육및사회용",
    "28000": "기타",
    "28001": "기타",
    "28002": "기타",
    "28003": "기타",
    "28004": "기타",
    "28005": "기타",
    "28006": "기타",
    "28007": "기타",
    "28008": "기타",
    "28009": "기타",
    "28010": "기타",
    "28011": "기타",
    "28012": "기타",
    "28013": "기타",
    "28014": "기타",
    "28015": "기타",
    "28016": "기타",
    "28017": "기타",
    "28018": "기타",
    "28019": "기타",
    "28020": "기타",
    "28021": "기타",
    "28022": "기타",
    "28023": "기타",
    "28999": "기타",
    "29000": "교육및사회용",
    "29001": "교육및사회용",
    "29002": "교육및사회용",
    "30000": "기타",
    "30001": "기타",
    "30002": "기타",
    "30003": "기타",
    "30004": "기타",
    "30005": "기타",
    "30999": "기타",
    "31000": "상업용",
    "31001": "상업용",
    "31002": "상업용",
    "31003": "상업용",
    "31004": "상업용",
    "31005": "상업용",
    "31999": "상업용",
    "32000": "기타",
    "32001": "기타",
    "32002": "기타",
    "32003": "기타",
    "32004": "기타",
    "32005": "기타",
    "32100": "기타",
    "32101": "기타",
    "32102": "기타",
    "32103": "기타",
    "32999": "기타",
    "33000": "기타",
    "33001": "기타",
    "33999": "기타",
    "Z0000": "기타",
    "Z3000": "상업용",
    "Z3001": "상업용",
    "Z3002": "상업용",
    "Z3003": "상업용",
    "Z3004": "상업용",
    "Z3005": "상업용",
    "Z3006": "상업용",
    "Z3007": "상업용",
    "Z3008": "상업용",
    "Z3009": "상업용",
    "Z3010": "상업용",
    "Z3011": "상업용",
    "Z3012": "상업용",
    "Z3014": "상업용",
    "Z3015": "상업용",
    "Z3016": "상업용",
    "Z3017": "상업용",
    "Z3018": "상업용",
    "Z3019": "상업용",
    "Z3020": "상업용",
    "Z3021": "상업용",
    "Z3022": "상업용",
    "Z3023": "상업용",
    "Z3100": "기타",
    "Z3101": "기타",
    "Z3102": "기타",
    "Z3103": "기타",
    "Z3104": "기타",
    "Z3105": "기타",
    "Z3106": "기타",
    "Z3107": "기타",
    "Z3108": "기타",
    "Z3109": "기타",
    "Z3199": "기타",
    "Z3201": "상업용",
    "Z3202": "상업용",
    "Z3203": "상업용",
    "Z3204": "상업용",
    "Z3205": "상업용",
    "Z3206": "상업용",
    "Z3207": "상업용",
    "Z3208": "상업용",
    "Z3209": "상업용",
    "Z3210": "상업용",
    "Z3211": "상업용",
    "Z3212": "상업용",
    "Z3214": "상업용",
    "Z3215": "상업용",
    "Z3216": "상업용",
    "Z3217": "상업용",
    "Z3218": "상업용",
    "Z3219": "상업용",
    "Z3220": "상업용",
    "Z3221": "상업용",
    "Z3300": "교육및사회용",
    "Z3301": "교육및사회용",
    "Z3302": "교육및사회용",
    "Z3303": "교육및사회용",
    "Z3304": "교육및사회용",
    "Z3305": "교육및사회용",
    "Z3306": "교육및사회용",
    "Z3307": "교육및사회용",
    "Z3399": "교육및사회용",
    "Z3400": "교육및사회용",
    "Z3401": "교육및사회용",
    "Z3402": "교육및사회용",
    "Z3403": "교육및사회용",
    "Z3499": "교육및사회용",
    "Z3500": "교육및사회용",
    "Z3501": "교육및사회용",
    "Z3502": "교육및사회용",
    "Z3503": "교육및사회용",
    "Z3599": "교육및사회용",
    "Z3600": "상업용",
    "Z3601": "상업용",
    "Z3602": "상업용",
    "Z3603": "상업용",
    "Z3604": "상업용",
    "Z3699": "상업용",
    "Z3999": "상업용",
    "Z5000": "교육및사회용",
    "Z6000": "상업용",
    "Z6205": "상업용",
    "Z6999": "상업용",
    "Z7001": "기타",
    "Z7002": "기타",
    "Z8000": "교육및사회용",
    "Z8999": "교육및사회용",
    "Z9000": "기타",
    "Z9001": "기타",
    "Z9999": "기타",
}

건축법 제2조 제1항 제11호 나목

시장ㆍ군수ㆍ구청장(자치구의 구청장을 말한다. 이하 같다)

제11조(건축허가) ① 건축물을 건축하거나 대수선하려는 자는 특별자치시장ㆍ특별자치도지사 또는 시장ㆍ군수ㆍ구청장의 허가를 받아야 한다.


자치구가 아닌 구를 제외한 집계단위는 나중에 따로 사용


In [7]:
sigungu_autonomous_agg = {
    "11110": "11110",
    "11140": "11140",
    "11170": "11170",
    "11200": "11200",
    "11215": "11215",
    "11230": "11230",
    "11260": "11260",
    "11290": "11290",
    "11305": "11305",
    "11320": "11320",
    "11350": "11350",
    "11380": "11380",
    "11410": "11410",
    "11440": "11440",
    "11470": "11470",
    "11500": "11500",
    "11530": "11530",
    "11545": "11545",
    "11560": "11560",
    "11590": "11590",
    "11620": "11620",
    "11650": "11650",
    "11680": "11680",
    "11710": "11710",
    "11740": "11740",
    "26110": "26110",
    "26140": "26140",
    "26170": "26170",
    "26200": "26200",
    "26230": "26230",
    "26260": "26260",
    "26290": "26290",
    "26320": "26320",
    "26350": "26350",
    "26380": "26380",
    "26410": "26410",
    "26440": "26440",
    "26470": "26470",
    "26500": "26500",
    "26530": "26530",
    "26710": "26710",
    "27110": "27110",
    "27140": "27140",
    "27170": "27170",
    "27200": "27200",
    "27230": "27230",
    "27260": "27260",
    "27290": "27290",
    "27710": "27710",
    "28110": "28110",
    "28140": "28140",
    "28170": "28177",
    "28177": "28177",
    "28185": "28185",
    "28200": "28200",
    "28237": "28237",
    "28245": "28245",
    "28260": "28260",
    "28710": "28710",
    "28720": "28720",
    "29110": "29110",
    "29140": "29140",
    "29155": "29155",
    "29170": "29170",
    "29200": "29200",
    "30110": "30110",
    "30140": "30140",
    "30170": "30170",
    "30200": "30200",
    "30230": "30230",
    "31110": "31110",
    "31140": "31140",
    "31170": "31170",
    "31200": "31200",
    "31710": "31710",
    "36110": "36110",
    "41111": "41110",
    "41113": "41110",
    "41115": "41110",
    "41117": "41110",
    "41131": "41130",
    "41133": "41130",
    "41135": "41130",
    "41150": "41150",
    "41171": "41170",
    "41173": "41170",
    "41190": "41190",
    "41210": "41210",
    "41220": "41220",
    "41250": "41250",
    "41271": "41270",
    "41273": "41270",
    "41281": "41280",
    "41283": "41280",
    "41285": "41280",
    "41287": "41280",
    "41290": "41290",
    "41310": "41310",
    "41360": "41360",
    "41370": "41370",
    "41390": "41390",
    "41410": "41410",
    "41430": "41430",
    "41450": "41450",
    "41461": "41460",
    "41463": "41460",
    "41465": "41460",
    "41480": "41480",
    "41500": "41500",
    "41550": "41550",
    "41570": "41570",
    "41590": "41590",
    "41610": "41610",
    "41630": "41630",
    "41650": "41650",
    "41670": "41670",
    "41800": "41800",
    "41820": "41820",
    "41830": "41830",
    "42110": "42110",
    "42130": "42130",
    "42150": "42150",
    "42170": "42170",
    "42190": "42190",
    "42210": "42210",
    "42230": "42230",
    "42720": "42720",
    "42730": "42730",
    "42750": "42750",
    "42760": "42760",
    "42770": "42770",
    "42780": "42780",
    "42790": "42790",
    "42800": "42800",
    "42810": "42810",
    "42820": "42820",
    "42830": "42830",
    "43111": "43110",
    "43112": "43110",
    "43113": "43110",
    "43114": "43110",
    "43130": "43130",
    "43150": "43150",
    "43720": "43720",
    "43730": "43730",
    "43740": "43740",
    "43745": "43745",
    "43750": "43750",
    "43760": "43760",
    "43770": "43770",
    "43800": "43800",
    "44130": "44130",
    "44131": "44130",
    "44133": "44130",
    "44150": "44150",
    "44180": "44180",
    "44200": "44200",
    "44210": "44210",
    "44230": "44230",
    "44250": "44250",
    "44270": "44270",
    "44710": "44710",
    "44760": "44760",
    "44770": "44770",
    "44790": "44790",
    "44800": "44800",
    "44810": "44810",
    "44825": "44825",
    "45111": "45110",
    "45113": "45110",
    "45130": "45130",
    "45140": "45140",
    "45180": "45180",
    "45190": "45190",
    "45210": "45210",
    "45710": "45710",
    "45720": "45720",
    "45730": "45730",
    "45740": "45740",
    "45750": "45750",
    "45770": "45770",
    "45790": "45790",
    "45800": "45800",
    "46110": "46110",
    "46130": "46130",
    "46150": "46150",
    "46170": "46170",
    "46230": "46230",
    "46710": "46710",
    "46720": "46720",
    "46730": "46730",
    "46770": "46770",
    "46780": "46780",
    "46790": "46790",
    "46800": "46800",
    "46810": "46810",
    "46820": "46820",
    "46830": "46830",
    "46840": "46840",
    "46860": "46860",
    "46870": "46870",
    "46880": "46880",
    "46890": "46890",
    "46900": "46900",
    "46910": "46910",
    "47111": "47110",
    "47113": "47110",
    "47130": "47130",
    "47150": "47150",
    "47170": "47170",
    "47190": "47190",
    "47210": "47210",
    "47230": "47230",
    "47250": "47250",
    "47280": "47280",
    "47290": "47290",
    "47720": "47720",
    "47730": "47730",
    "47750": "47750",
    "47760": "47760",
    "47770": "47770",
    "47820": "47820",
    "47830": "47830",
    "47840": "47840",
    "47850": "47850",
    "47900": "47900",
    "47920": "47920",
    "47930": "47930",
    "47940": "47940",
    "48121": "48120",
    "48123": "48120",
    "48125": "48120",
    "48127": "48120",
    "48129": "48120",
    "48170": "48170",
    "48220": "48220",
    "48240": "48240",
    "48250": "48250",
    "48270": "48270",
    "48310": "48310",
    "48330": "48330",
    "48720": "48720",
    "48730": "48730",
    "48740": "48740",
    "48820": "48820",
    "48840": "48840",
    "48850": "48850",
    "48860": "48860",
    "48870": "48870",
    "48880": "48880",
    "48890": "48890",
    "50110": "50110",
    "50130": "50130",
}

하지만 주민등록인구의 시군구 인구를 확인하고 단위를 맞출 필요는 있음. 교정이 필요한 경우에만 시군구 코드 변경(일부는 err로 집계)


In [8]:
class smart_dict(dict):
    def __missing__(self, key):
        return key


sigungu_fix = smart_dict(
    {
        "28170": "28177",  # 인천시 남구 미추홀구로 바뀌었는데 일부 남음
        "41283": "err",  # 경기도 고양시 일산구 분구되었는데 일부 남음
        "44130": "err",  # 천안시 일부 남음
    }
)

sigungu_agg = {
    key: sigungu_fix[key]
    for key in df_floor.select("시군구_코드")
    .unique()
    .sort("시군구_코드")
    .collect()["시군구_코드"]
    .to_list()
}

print(sigungu_agg["11110"])
print(sigungu_agg["28170"])
print(sigungu_agg["44130"])

11110
28177
err


## 층별개요 데이터 정제


In [9]:
df_floor.columns

['대지_위치',
 '도로명_대지_위치',
 '건물_명',
 '시군구_코드',
 '법정동_코드',
 '대지_구분_코드',
 '번',
 '지',
 '특수지_명',
 '블록',
 '로트',
 '새주소_도로_코드',
 '새주소_법정동_코드',
 '새주소_지상지하_코드',
 '새주소_본_번',
 '새주소_부_번',
 '동_명',
 '층_구분_코드',
 '층_구분_코드_명',
 '층_번호',
 '층_번호_명',
 '구조_코드',
 '구조_코드_명',
 '기타_구조',
 '주_용도_코드',
 '주_용도_코드_명',
 '기타_용도',
 '면적(㎡)',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '면적_제외_여부',
 '생성_일자',
 '관리_건축물대장_PK']

In [10]:
df_floor_selected = df_floor.select(
    [
        "시군구_코드",
        "층_구분_코드",
        "층_구분_코드_명",
        "층_번호",
        "층_번호_명",
        "주_용도_코드",
        "주_용도_코드_명",
        "면적(㎡)",
        "주_부속_구분_코드",
        "주_부속_구분_코드_명",
        "면적_제외_여부",
        "생성_일자",
        "관리_건축물대장_PK",
    ]
)

시군구 코드, 층 구분 코드, 주 용도 코드, 면적 (0 초과)은 있어야 하는 것인데, 없는 경우는 제외하였다.


In [11]:
df_floor_selected.count().collect()

시군구_코드,층_구분_코드,층_구분_코드_명,층_번호,층_번호_명,주_용도_코드,주_용도_코드_명,면적(㎡),주_부속_구분_코드,주_부속_구분_코드_명,면적_제외_여부,생성_일자,관리_건축물대장_PK
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
20670074,20669516,20669516,20670074,20666793,20644000,20641697,20670074,20660591,20660591,7716626,20670074,20670074


설명: 층 구분 코드가 없으면 주 용도 코드도 없고 면적도 0이고, 하나만 하지 않는다...


In [12]:
df_floor_selected.filter(pl.col("층_구분_코드").is_null()).head().collect()

시군구_코드,층_구분_코드,층_구분_코드_명,층_번호,층_번호_명,주_용도_코드,주_용도_코드_명,면적(㎡),주_부속_구분_코드,주_부속_구분_코드_명,면적_제외_여부,생성_일자,관리_건축물대장_PK
str,str,str,i64,str,str,str,f64,str,str,str,str,str
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188912"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188913"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188914"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188915"""
"""11170""",null,null,0,"""(내)""",null,null,0.0,"""0""","""주건축물""","""1""","""20200427""","""11170-100188916"""


층별개요 데이터에서 시군구*코드, 층*구분*코드, 주*용도\_코드 컬럼의 값이 존재하고(null이 아니고), 면적(㎡) 컬럼의 값이 0보다 큰 유효한 경우만 추출하였다.


In [13]:
df_filtered = df_floor_selected.filter(
    pl.col("시군구_코드").is_not_null()
    & pl.col("층_구분_코드").is_not_null()
    & pl.col("주_용도_코드").is_not_null()
    & (pl.col("면적(㎡)") > 0)
)

In [14]:
# df_filtered.count().collect()

## 표제부 데이터 연계


In [15]:
df_dong.columns

['대장_구분_코드',
 '대장_구분_코드_명',
 '대장_종류_코드',
 '대장_종류_코드_명',
 '대지_위치',
 '도로명_대지_위치',
 '건물_명',
 '시군구_코드',
 '법정동_코드',
 '대지_구분_코드',
 '번',
 '지',
 '특수지_명',
 '블록',
 '로트',
 '외필지_수',
 '새주소_도로_코드',
 '새주소_법정동_코드',
 '새주소_지상지하_코드',
 '새주소_본_번',
 '새주소_부_번',
 '동_명',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '대지_면적(㎡)',
 '건축_면적(㎡)',
 '건폐_율(%)',
 '연면적(㎡)',
 '용적_률_산정_연면적(㎡)',
 '용적_률(%)',
 '구조_코드',
 '구조_코드_명',
 '기타_구조',
 '주_용도_코드',
 '주_용도_코드_명',
 '기타_용도',
 '지붕_코드',
 '지붕_코드_명',
 '기타_지붕',
 '세대_수(세대)',
 '가구_수(가구)',
 '높이(m)',
 '지상_층_수',
 '지하_층_수',
 '승용_승강기_수',
 '비상용_승강기_수',
 '부속_건축물_수',
 '부속_건축물_면적(㎡)',
 '총_동_연면적(㎡)',
 '옥내_기계식_대수(대)',
 '옥내_기계식_면적(㎡)',
 '옥외_기계식_대수(대)',
 '옥외_기계식_면적(㎡)',
 '옥내_자주식_대수(대)',
 '옥내_자주식_면적(㎡)',
 '옥외_자주식_대수(대)',
 '옥외_자주식_면적(㎡)',
 '허가_일',
 '착공_일',
 '사용승인_일',
 '허가번호_년',
 '허가번호_기관_코드',
 '허가번호_기관_코드_명',
 '허가번호_구분_코드',
 '허가번호_구분_코드_명',
 '호_수(호)',
 '에너지효율_등급',
 '에너지절감_율',
 '에너지_EPI점수',
 '친환경_건축물_등급',
 '친환경_건축물_인증점수',
 '지능형_건축물_등급',
 '지능형_건축물_인증점수',
 '생성_일자',
 '내진_설계_적용_여부',
 '내진_능력',
 '관리_건축물대장_PK']

In [16]:
df_dong_selected = df_dong.select(
    [
        "시군구_코드",
        "주_부속_구분_코드",
        "주_부속_구분_코드_명",
        "대지_면적(㎡)",
        "건축_면적(㎡)",
        "연면적(㎡)",
        "용적_률_산정_연면적(㎡)",
        "주_용도_코드",
        "주_용도_코드_명",
        "사용승인_일",
        "관리_건축물대장_PK",
    ]
)

1. 전체 일반건축물/집합건축물의 표제부 동 수: 7,943,168 동
2. 주건축물 7,308,393 동, 부속건축물 634,775 동
3. 연면적 기재 7,879,427 동


In [17]:
df_dong_selected.count().collect()

시군구_코드,주_부속_구분_코드,주_부속_구분_코드_명,대지_면적(㎡),건축_면적(㎡),연면적(㎡),용적_률_산정_연면적(㎡),주_용도_코드,주_용도_코드_명,사용승인_일,관리_건축물대장_PK
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
7943168,7942481,7942481,7943168,7943168,7943168,7943168,7912311,7912173,7270272,7943168


In [18]:
df_dong_selected.select("주_부속_구분_코드").group_by(
    "주_부속_구분_코드"
).len().collect()

주_부속_구분_코드,len
str,u32
"""1""",634775
null,687
"""0""",7307706


연면적에 (0이 아닌) 값이 가장 많기 때문에 연면적을 기준으로 정제 (계획)


In [19]:
df_dong_selected.select(
    [
        (pl.col("대지_면적(㎡)") > 0).sum().alias("대지_면적(㎡) > 0"),
        (pl.col("건축_면적(㎡)") > 0).sum().alias("건축_면적(㎡) > 0"),
        (pl.col("연면적(㎡)") > 0).sum().alias("연면적(㎡) > 0"),
        (pl.col("용적_률_산정_연면적(㎡)") > 0)
        .sum()
        .alias("용적_률_산정_연면적(㎡) > 0"),
    ]
).collect()

대지_면적(㎡) > 0,건축_면적(㎡) > 0,연면적(㎡) > 0,용적_률_산정_연면적(㎡) > 0
u32,u32,u32,u32
4160181,7452568,7879427,7462902


표제부 기준, 2022년 말 이전 사용승인된 주건축물만 추출: 7,307,806 동


In [20]:
df_dong_filtered = (
    df_dong_selected.filter(
        (pl.col("주_부속_구분_코드") == "0") | (pl.col("주_부속_구분_코드").is_null())
    )
    .filter(
        (pl.col("사용승인_일").str.len_chars() != 8)
        | (pl.col("사용승인_일").fill_null("19000101") <= "20221231")
    )
    .select(["연면적(㎡)", "관리_건축물대장_PK"])
)

In [21]:
df_dong_filtered.count().collect()

연면적(㎡),관리_건축물대장_PK
u32,u32
7307806,7307806


join


In [22]:
df_joined = df_filtered.join(df_dong_filtered, on="관리_건축물대장_PK", how="inner")

In [23]:
df_joined.select(pl.len()).collect()

len
u32
19890013


In [24]:
df_joined.filter(
    (pl.col("면적(㎡)") > pl.col("연면적(㎡)")) & (pl.col("연면적(㎡)") > 0)
).select(pl.len()).collect()

len
u32
9661


In [25]:
# df_joined.filter((pl.col("면적(㎡)") > pl.col("연면적(㎡)"))).head().collect()

연면적 기준 정제(실행): 층별면적이 연면적보다 작은 것만 남기는 것을 원칙으로 하지만, 연면적이 0 이하인 오류인 경우 제거하지 않음.


In [26]:
df_joined2 = df_joined.filter(
    (pl.col("면적(㎡)") <= pl.col("연면적(㎡)")) | (pl.col("연면적(㎡)") <= 0)
)

-   전체 층별개요 데이터: 20,613,365 층
-   2022년 말 전국 건축물 층 수: 19,890,013 층
-   층 면적이 연면적보다 큰 경우(연면적 값이 존재하는 경우에 한정): 9,661 층
-   주건축물의 유효한 층 수: 19,880,352 층


In [27]:
df_joined2.select(pl.len()).collect()

len
u32
19880352


2022년 말 기준 전국 건축물의 지상 층수의 합계는 18,056,700개 층이며, 지하 층수의 합계는 1,415,933개 층이다. 기타로 옥탑은 407,556개가 있으며, 복수층(상층)은 5개, 복수층(하층)은 6개, 각층은 152개가 있다. 이를 모두 합한 전체 층의 합계는 19,880,352개 층이다.


In [28]:
df_joined2.group_by(["층_구분_코드", "층_구분_코드_명"]).len().collect()

층_구분_코드,층_구분_코드_명,len
str,str,u32
"""40""","""각층""",152
"""20""","""지상""",18056700
"""10""","""지하""",1415933
"""22""","""복수층(상층)""",5
"""21""","""복수층(하층)""",6
"""30""","""옥탑""",407556


## 집계


In [29]:
df_sgg.head().collect()

시군구코드,시군구명,폐지여부
str,str,str
"""11000""","""서울특별시""","""존재"""
"""11110""","""서울특별시 종로구""","""존재"""
"""11140""","""서울특별시 중구""","""존재"""
"""11170""","""서울특별시 용산구""","""존재"""
"""11200""","""서울특별시 성동구""","""존재"""


In [30]:
df_joined2.columns

['시군구_코드',
 '층_구분_코드',
 '층_구분_코드_명',
 '층_번호',
 '층_번호_명',
 '주_용도_코드',
 '주_용도_코드_명',
 '면적(㎡)',
 '주_부속_구분_코드',
 '주_부속_구분_코드_명',
 '면적_제외_여부',
 '생성_일자',
 '관리_건축물대장_PK',
 '연면적(㎡)']

replace 메소드의 default 인자 사용하면 됨. 따로 만들어 쓸 필요는 없음.


In [31]:
# Function to map with a default value
# def map_with_default(value, mapping, default):
#     return mapping.get(value, default)

시군구 오류 수정, 용도 4대 용도 + 기타로 집계.


In [32]:
df_joined3 = df_joined2.select(
    ["시군구_코드", "주_용도_코드", "주_용도_코드_명", "면적(㎡)"]
).with_columns(
    pl.col("시군구_코드").replace(sigungu_agg, default="err").alias("시군구_집계단위"),
    pl.col("주_용도_코드").replace(use_agg, default="기타").alias("용도_집계단위"),
)

df_joined3.select(pl.len()).collect()

len
u32
19880352


In [33]:
df_joined3.filter(pl.col("시군구_코드") == "41111").head().collect()

시군구_코드,주_용도_코드,주_용도_코드_명,면적(㎡),시군구_집계단위,용도_집계단위
str,str,str,f64,str,str
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""주거용"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""주거용"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""주거용"""
"""41111""","""02003""","""다세대주택""",99.6,"""41111""","""주거용"""
"""41111""","""06101""","""교회""",367.29,"""41111""","""교육및사회용"""


In [34]:
df_aggregated = df_joined3.group_by(["시군구_집계단위", "용도_집계단위"]).agg(
    pl.sum("면적(㎡)").cast(pl.Int64).alias("면적_합계(㎡)")
)

In [35]:
df_pivot = (
    df_aggregated.collect()
    .pivot(values="면적_합계(㎡)", index="시군구_집계단위", columns="용도_집계단위")
    .join(
        df_sgg.select(["시군구코드", "시군구명"]).collect(),
        left_on="시군구_집계단위",
        right_on="시군구코드",
        how="left",
    )
    .sort("시군구_집계단위")
    .select(
        "시군구_집계단위",
        "시군구명",
        "주거용",
        "상업용",
        "공업용",
        "교육및사회용",
        "기타",
    )
)
df_pivot.filter(pl.col("시군구_집계단위").str.starts_with("41"))

시군구_집계단위,시군구명,주거용,상업용,공업용,교육및사회용,기타
str,str,i64,i64,i64,i64,i64
"""41111""","""경기도 수원시 장안구""",8993945,2421433,206243,1886036,323440
"""41113""","""경기도 수원시 권선구""",12810670,5761416,1352945,1443347,348592
"""41115""","""경기도 수원시 팔달구""",6001188,5118233,15693,1253431,273313
"""41117""","""경기도 수원시 영통구""",12567345,5614601,2193054,3565236,190180
"""41131""","""경기도 성남시 수정구""",6790604,3933813,268007,1710267,597204
…,…,…,…,…,…,…
"""41650""","""경기도 포천시""",4862716,3700799,6403084,1491224,3680305
"""41670""","""경기도 여주시""",4640227,2017410,1473341,1257445,3431935
"""41800""","""경기도 연천군""",1388840,759203,375103,326438,1922380


주거, 상업, 공업, 교육및사회 용도는 얼추 비슷하게 나오나, 기타 용도는 크게 차이가 나기도 함.


In [36]:
df_pivot.write_csv("output/floor_area_by_sgg_orig_and_use_agg.csv", include_bom=True)

In [38]:
from typing import Union


def floor_area_by_sgg_use(df: pl.DataFrame, sigungu_agg, use_agg) -> pl.DataFrame:
    df = df.select(
        ["시군구_코드", "주_용도_코드", "주_용도_코드_명", "면적(㎡)"]
    ).with_columns(
        pl.col("시군구_코드")
        .replace(sigungu_agg, default="err")
        .alias("시군구_집계단위"),
        pl.col("주_용도_코드").replace(use_agg, default="기타").alias("용도_집계단위"),
    )
    df_aggregated = df.group_by(["시군구_집계단위", "용도_집계단위"]).agg(
        pl.sum("면적(㎡)").cast(pl.Int64).alias("면적_합계(㎡)")
    )
    df_pivot = (
        df_aggregated.collect()
        .pivot(values="면적_합계(㎡)", index="시군구_집계단위", columns="용도_집계단위")
        .join(
            df_sgg.select(["시군구코드", "시군구명"]).collect(),
            left_on="시군구_집계단위",
            right_on="시군구코드",
            how="left",
        )
        .sort("시군구_집계단위")
        .select(
            "시군구_집계단위",
            "시군구명",
            "주거용",
            "상업용",
            "공업용",
            "교육및사회용",
            "기타",
        )
    )
    return df_pivot


floor_area_by_sgg_use(df_joined2, sigungu_autonomous_agg, use_agg).write_csv(
    "output/floor_area_by_sgg_autonomous_and_use_agg.csv", include_bom=True
)
